In [ ]:
# Cell 1: Complete Self-Contained Visualization Setup

# 1. Import Sedona Context and the updated Maps module to avoid deprecation warnings
from sedona.spark import *
from sedona.spark.maps.SedonaKepler import SedonaKepler

print("Initializing Sedona Context...")
# 2. Build or retrieve the active Spark session managed by Wherobots
config = SedonaContext.builder().getOrCreate()
sedona = SedonaContext.create(config)

print("Extracting data samples for visualization...")
try:
    # 3. Query samples directly from your Iceberg catalog tables
    # Limiting to 2000 keeps the browser responsive when rendering polygons
    demo_df = sedona.sql("SELECT * FROM org_catalog.fgsdb.abs_demographics LIMIT 2000")
    stations_df = sedona.sql("SELECT * FROM org_catalog.fgsdb.nsw_rail_stations")

    print(f"Loaded {demo_df.count()} demographic rows and {stations_df.count()} station rows.")

    # 4. Initialize the Kepler.gl map with the demographic layer
    print("Generating map canvas...")
    # Center the map viewport on the Newcastle / Lake Macquarie area in Australia
    map_config = {
        "version": "v1",
        "config": {
            "mapState": {
                "latitude": -32.9283,
                "longitude": 151.7817,
                "zoom": 10,
                "bearing": 0,
                "pitch": 0,
                "dragRotate": false
            }
        }
    }
    map_vis = SedonaKepler.create_map(df=demo_df, name="Raw Demographics (Sample)", config=map_config)

    # 5. Overlay the transit stations layer onto the map
    SedonaKepler.add_df(map_vis, df=stations_df, name="NSW Rail Stations")

    # 6. Export the interactive map as a standalone HTML file in your Jupyter workspace
    output_file = "raw_data_explorer.html"
    map_vis.save_to_html(file_name=output_file)

    print(f"\n🚀 Success! Map saved to your workspace as: {output_file}")
    print("Refresh your file explorer side-panel, download 'raw_data_explorer.html', and open it locally.")

except Exception as e:
    print(f"\n❌ Execution failed: {str(e)}")
    print("Please verify that the tables exist in 'org_catalog.fgsdb' and that your notebook runtime is active.")